In [ ]:
!pip install -qU langchain langchain-core langgraph langchain-groq langchain-community duckduckgo-search langgraph-checkpoint-mongodb langchain_groq

In [1]:
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END, add_messages
from typing_extensions import Annotated, TypedDict, Literal
from rich import print
from langchain_groq import ChatGroq
from langgraph.checkpoint.mongodb import MongoDBSaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langgraph.prebuilt import tool_node, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.types import Command, interrupt
from typing_extensions import Optional

In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm = ChatGroq(model="groq/compound-mini", temperature=0.5)

In [4]:
class State(TypedDict):
    info: str
    draft: str
    human_feedback: Optional[str]
    is_approved: Optional[str]
    final_post: Optional[str]


def generate_draft(state: State) -> dict:
    info = state["info"]
    prompt = f"""You are a social media manager. Create a short and engaging social media post based on the following information:
    Topic: {info.get('topic')}
    Key Points: {info.get('key_points')}
    Tone: {info.get('tone')}
    Target Audience: {info.get('target_audience')}
    Write an linkedin style of post of (1-2 Paragraphs) by using this information.
    """

    response = llm.invoke(prompt)
    print("Generating the draft...\n")
    print("Generated Draft:\n", response.content)
    draft = response.content
    return {"draft": draft, "approved": False}

def ask_for_feedback(state: State) -> dict:
    print("\nAsking for human feedback...\n\n")
    print("Draft for review:\n", state["draft"])
    feedback = interrupt("Please provide your feedback on the draft. 'Approved' if you like it! ")
    is_approved = feedback.strip().lower() == "approved"
    return {"human_feedback": feedback, "is_approved": is_approved}

def decide_next_step(state: State) -> Literal["postToLinkedIn", "revise_draft"]:
    if state["is_approved"]:
        return "postToLinkedIn"
    else:
        return "revise_draft"

def revise_draft(state: State) -> dict:
    feedback = state["human_feedback"] or ""
    prompt = f"""You are a social media manager. Revise the following draft based on the human feedback provided {feedback}.
    Old Draft: {state["draft"]}
    Please provide a revised version of the draft. Keep the same tone and style but make improvements based on the feedback.
    """
    print("\nRevising the draft based on feedback...\n\n")
    response = llm.invoke(prompt)
    print("Revised Draft:\n", response.content)
    return {"draft": response.content, "approved": False}

def postToLinkedIn(state: State) -> dict:
    finalDraft = state["draft"]
    print("\nPosting to LinkedIn...\n\n")
    print("Final Post:\n", finalDraft)
    return {"final_post": finalDraft}

In [5]:
graph_builder = StateGraph(State)

graph_builder.add_node("generate_draft", generate_draft)
graph_builder.add_node("ask_for_feedback", ask_for_feedback)
graph_builder.add_node("decide_next_step", decide_next_step)
graph_builder.add_node("revise_draft", revise_draft)
graph_builder.add_node("postToLinkedIn", postToLinkedIn)

graph_builder.add_edge(START, "generate_draft")
graph_builder.add_edge("generate_draft", "ask_for_feedback")
graph_builder.add_conditional_edges("ask_for_feedback", decide_next_step)
graph_builder.add_edge("revise_draft", "ask_for_feedback")
graph_builder.add_edge("postToLinkedIn", END)

def graph_with_checkpointing(checkpointer):
    graph = graph_builder.compile(checkpointer)
    return graph


checkpointer = InMemorySaver()
graph = graph_with_checkpointing(checkpointer)

config = {
            "configurable": {
                "thread_id": "Mukaram"    # User ID
            }
        }

In [6]:
print(graph.invoke(State({"info": {
    "topic": "The Benefits of Remote Work",
    "key_points": ["Increased flexibility", "Reduced commute time", "Access to global talent"],
    "tone": "Professional",
    "target_audience": "Remote workers and HR professionals"
}}), config))
print("State Snapshots\n")
print(graph.get_state(config))

Generating the draft...

Generated Draft:
 🚀 Embracing remote work isn’t just a trend—it’s a strategic advantage. By giving employees **increased 
flexibility**, organizations empower teams to balance productivity with personal well‑being, driving higher 
engagement and results. Cutting out the daily commute translates into **saved time and reduced stress**, allowing 
talent to focus on what truly matters. Plus, a remote‑first model opens the door to **global talent**, expanding 
your talent pool beyond geographic limits and fostering diverse, innovative perspectives.  

For HR leaders and remote professionals alike, the message is clear: leverage flexibility, eliminate unnecessary 
travel, and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼 #RemoteWork 
#FutureOfWork #HRStrategy #TalentAcquisition

Asking for human feedback...

Draft for review:
 🚀 Embracing remote work isn’t just a trend—it’s a strategic advantage. By giving employees **increased 
flexibility**, organizations empower teams to balance productivity with personal well‑being, driving higher 
engagement and results. Cutting out the daily commute translates into **saved time and reduced stress**, allowing 
talent to focus on what truly matters. Plus, a remote‑first model opens the door to **global talent**, expanding 
your talent pool beyond geographic limits and fostering diverse, innovative perspectives.  

For HR leaders and remote professionals alike, the message is clear: leverage flexibility, eliminate unnecessary 
travel, and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼 #RemoteWork 
#FutureOfWork #HRStrategy #TalentAcquisition

{
    'info': {
        'topic': 'The Benefits of Remote Work',
        'key_points': ['Increased flexibility', 'Reduced commute time', 'Access to global talent'],
        'tone': 'Professional',
        'target_audience': 'Remote workers and HR professionals'
    },
    'draft': '🚀 Embracing remote work isn’t just a trend—it’s a strategic advantage. By giving employees 
**increased flexibility**, organizations empower teams to balance productivity with personal well‑being, driving 
higher engagement and results. Cutting out the daily commute translates into **saved time and reduced stress**, 
allowing talent to focus on what truly matters. Plus, a remote‑first model opens the door to **global talent**, 
expanding your talent pool beyond geographic limits and fostering diverse, innovative perspectives.  \n\nFor HR 
leaders and remote professionals alike, the message is clear: leverage flexibility, eliminate unnecessary travel, 
and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼 #RemoteWork #FutureOfWork 
#HRStrategy #TalentAcquisition',
    '__interrupt__': [
        Interrupt(
            value="Please provide your feedback on the draft. 'Approved' if you like it! ",
            id='56fca0aafcb4eef5192b8120e43a8e5a'
        )
    ]
}

State Snapshots

StateSnapshot(
    values={
        'info': {
            'topic': 'The Benefits of Remote Work',
            'key_points': ['Increased flexibility', 'Reduced commute time', 'Access to global talent'],
            'tone': 'Professional',
            'target_audience': 'Remote workers and HR professionals'
        },
        'draft': '🚀 Embracing remote work isn’t just a trend—it’s a strategic advantage. By giving employees 
**increased flexibility**, organizations empower teams to balance productivity with personal well‑being, driving 
higher engagement and results. Cutting out the daily commute translates into **saved time and reduced stress**, 
allowing talent to focus on what truly matters. Plus, a remote‑first model opens the door to **global talent**, 
expanding your talent pool beyond geographic limits and fostering diverse, innovative perspectives.  \n\nFor HR 
leaders and remote professionals alike, the message is clear: leverage flexibility, eliminate unnecessary travel, 
and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼 #RemoteWork #FutureOfWork 
#HRStrategy #TalentAcquisition'
    },
    next=('ask_for_feedback',),
    config={
        'configurable': {
            'thread_id': 'Mukaram',
            'checkpoint_ns': '',
            'checkpoint_id': '1f13ce6f-18a3-6e93-8001-686d7b2cc436'
        }
    },
    metadata={'source': 'loop', 'step': 1, 'parents': {}},
    created_at='2026-04-20T18:30:01.770946+00:00',
    parent_config={
        'configurable': {
            'thread_id': 'Mukaram',
            'checkpoint_ns': '',
            'checkpoint_id': '1f13ce6f-090d-6e2d-8000-302bb60d729c'
        }
    },
    tasks=(
        PregelTask(
            id='85f1ca68-62bb-3560-b869-a033c36b0b1e',
            name='ask_for_feedback',
            path=('__pregel_pull', 'ask_for_feedback'),
            error=None,
            interrupts=(
                Interrupt(
                    value="Please provide your feedback on the draft. 'Approved' if you like it! ",
                    id='56fca0aafcb4eef5192b8120e43a8e5a'
                ),
            ),
            state=None,
            result=None
        ),
    ),
    interrupts=(
        Interrupt(
            value="Please provide your feedback on the draft. 'Approved' if you like it! ",
            id='56fca0aafcb4eef5192b8120e43a8e5a'
        ),
    )
)

In [7]:
state = graph.get_state(config)
interrupts = getattr(state, "interrupts", None) or []
if not interrupts:
    print("Unexpected! No Interrupt found!")
else:
  for pending_Interrupt in interrupts:
    print(f'Interrupt: {pending_Interrupt.id}\nValue: {pending_Interrupt.value}')

    userFeedback = input("Enter Feedback or type 'approved'!\n")

    print(graph.invoke(Command(resume={pending_Interrupt.id: userFeedback}), config))

    print(f"Graph after Resume with Human Feedback: \n{graph.get_state(config)}")




Interrupt: 56fca0aafcb4eef5192b8120e43a8e5a
Value: Please provide your feedback on the draft. 'Approved' if you like it!

Enter Feedback or type 'approved'!
generate another post


Asking for human feedback...

Draft for review:
 🚀 Embracing remote work isn’t just a trend—it’s a strategic advantage. By giving employees **increased 
flexibility**, organizations empower teams to balance productivity with personal well‑being, driving higher 
engagement and results. Cutting out the daily commute translates into **saved time and reduced stress**, allowing 
talent to focus on what truly matters. Plus, a remote‑first model opens the door to **global talent**, expanding 
your talent pool beyond geographic limits and fostering diverse, innovative perspectives.  

For HR leaders and remote professionals alike, the message is clear: leverage flexibility, eliminate unnecessary 
travel, and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼 #RemoteWork 
#FutureOfWork #HRStrategy #TalentAcquisition

Revising the draft based on feedback...

Revised Draft:
 🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  

When employees enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting 
engagement and results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what
truly matters.  

A remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  

**HR leaders & remote‑first professionals:** Leverage flexibility, ditch unnecessary travel, and tap into worldwide
expertise to build a resilient, future‑ready workforce. 🌍💼  

#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent

Asking for human feedback...

Draft for review:
 🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  

When employees enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting 
engagement and results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what
truly matters.  

A remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  

**HR leaders & remote‑first professionals:** Leverage flexibility, ditch unnecessary travel, and tap into worldwide
expertise to build a resilient, future‑ready workforce. 🌍💼  

#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent

{
    'info': {
        'topic': 'The Benefits of Remote Work',
        'key_points': ['Increased flexibility', 'Reduced commute time', 'Access to global talent'],
        'tone': 'Professional',
        'target_audience': 'Remote workers and HR professionals'
    },
    'draft': '🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  \n\nWhen employees enjoy **greater 
flexibility**, they can sync productivity with personal well‑being, boosting engagement and results. Eliminating 
the daily commute means **more time, less stress**, and a sharper focus on what truly matters.  \n\nA remote‑first 
approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing diverse, innovative
perspectives to the table.  \n\n**HR leaders & remote‑first professionals:**\u202fLeverage flexibility, ditch 
unnecessary travel, and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼  
\n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent',
    'human_feedback': 'generate another post',
    'is_approved': False,
    '__interrupt__': [
        Interrupt(
            value="Please provide your feedback on the draft. 'Approved' if you like it! ",
            id='e4bae188058d1e3b8648abcbc075cad3'
        )
    ]
}

Graph after Resume with Human Feedback: 
StateSnapshot(values={'info': {'topic': 'The Benefits of Remote Work', 'key_points': ['Increased flexibility', 
'Reduced commute time', 'Access to global talent'], 'tone': 'Professional', 'target_audience': 'Remote workers and 
HR professionals'}, 'draft': '🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  \n\nWhen employees 
enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting engagement and 
results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what truly matters.
\n\nA remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  \n\n**HR leaders & remote‑first professionals:**\u202fLeverage 
flexibility, ditch unnecessary travel, and tap into worldwide expertise to build a resilient, future‑ready 
workforce. 🌍💼  \n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent', 
'human_feedback': 'generate another post', 'is_approved': False}, next=('ask_for_feedback',), 
config={'configurable': {'thread_id': 'Mukaram', 'checkpoint_ns': '', 'checkpoint_id': 
'1f13ce72-5ede-607b-8003-773237151b1e'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, 
created_at='2026-04-20T18:31:29.665421+00:00', parent_config={'configurable': {'thread_id': 'Mukaram', 
'checkpoint_ns': '', 'checkpoint_id': '1f13ce72-44b2-6c36-8002-65ba53bedb5b'}}, 
tasks=(PregelTask(id='cf1c82e5-7e85-4220-87f1-500d0864134f', name='ask_for_feedback', path=('__pregel_pull', 
'ask_for_feedback'), error=None, interrupts=(Interrupt(value="Please provide your feedback on the draft. 'Approved'
if you like it! ", id='e4bae188058d1e3b8648abcbc075cad3'),), state=None, result=None),), 
interrupts=(Interrupt(value="Please provide your feedback on the draft. 'Approved' if you like it! ", 
id='e4bae188058d1e3b8648abcbc075cad3'),))

In [8]:
state = graph.get_state(config)
interrupts = getattr(state, "interrupts", None) or []
if not interrupts:
    print("Unexpected! No Interrupt found!")
else:
  for pending_Interrupt in interrupts:
    print(f'Interrupt: {pending_Interrupt.id}\nValue: {pending_Interrupt.value}')

    userFeedback = input("Enter Feedback or type 'approved'!\n")

    print(graph.invoke(Command(resume={pending_Interrupt.id: userFeedback}), config))

    print(f"Graph after Resume with Human Feedback: \n{graph.get_state(config)}")

Interrupt: e4bae188058d1e3b8648abcbc075cad3
Value: Please provide your feedback on the draft. 'Approved' if you like it!

Enter Feedback or type 'approved'!
approved


Asking for human feedback...

Draft for review:
 🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  

When employees enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting 
engagement and results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what
truly matters.  

A remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  

**HR leaders & remote‑first professionals:** Leverage flexibility, ditch unnecessary travel, and tap into worldwide
expertise to build a resilient, future‑ready workforce. 🌍💼  

#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent

Posting to LinkedIn...

Final Post:
 🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  

When employees enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting 
engagement and results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what
truly matters.  

A remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  

**HR leaders & remote‑first professionals:** Leverage flexibility, ditch unnecessary travel, and tap into worldwide
expertise to build a resilient, future‑ready workforce. 🌍💼  

#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent

{
    'info': {
        'topic': 'The Benefits of Remote Work',
        'key_points': ['Increased flexibility', 'Reduced commute time', 'Access to global talent'],
        'tone': 'Professional',
        'target_audience': 'Remote workers and HR professionals'
    },
    'draft': '🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  \n\nWhen employees enjoy **greater 
flexibility**, they can sync productivity with personal well‑being, boosting engagement and results. Eliminating 
the daily commute means **more time, less stress**, and a sharper focus on what truly matters.  \n\nA remote‑first 
approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing diverse, innovative
perspectives to the table.  \n\n**HR leaders & remote‑first professionals:**\u202fLeverage flexibility, ditch 
unnecessary travel, and tap into worldwide expertise to build a resilient, future‑ready workforce. 🌍💼  
\n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent',
    'human_feedback': 'approved',
    'is_approved': True,
    'final_post': '🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  \n\nWhen employees enjoy 
**greater flexibility**, they can sync productivity with personal well‑being, boosting engagement and results. 
Eliminating the daily commute means **more time, less stress**, and a sharper focus on what truly matters.  \n\nA 
remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  \n\n**HR leaders & remote‑first professionals:**\u202fLeverage 
flexibility, ditch unnecessary travel, and tap into worldwide expertise to build a resilient, future‑ready 
workforce. 🌍💼  \n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent'
}

Graph after Resume with Human Feedback: 
StateSnapshot(values={'info': {'topic': 'The Benefits of Remote Work', 'key_points': ['Increased flexibility', 
'Reduced commute time', 'Access to global talent'], 'tone': 'Professional', 'target_audience': 'Remote workers and 
HR professionals'}, 'draft': '🚀 **Remote work isn’t just a buzzword—it’s a strategic edge.**  \n\nWhen employees 
enjoy **greater flexibility**, they can sync productivity with personal well‑being, boosting engagement and 
results. Eliminating the daily commute means **more time, less stress**, and a sharper focus on what truly matters.
\n\nA remote‑first approach also unlocks **global talent**, expanding your talent pool beyond borders and bringing 
diverse, innovative perspectives to the table.  \n\n**HR leaders & remote‑first professionals:**\u202fLeverage 
flexibility, ditch unnecessary travel, and tap into worldwide expertise to build a resilient, future‑ready 
workforce. 🌍💼  \n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition #Flexibility #GlobalTalent', 
'human_feedback': 'approved', 'is_approved': True, 'final_post': '🚀 **Remote work isn’t just a buzzword—it’s a 
strategic edge.**  \n\nWhen employees enjoy **greater flexibility**, they can sync productivity with personal 
well‑being, boosting engagement and results. Eliminating the daily commute means **more time, less stress**, and a 
sharper focus on what truly matters.  \n\nA remote‑first approach also unlocks **global talent**, expanding your 
talent pool beyond borders and bringing diverse, innovative perspectives to the table.  \n\n**HR leaders & 
remote‑first professionals:**\u202fLeverage flexibility, ditch unnecessary travel, and tap into worldwide expertise
to build a resilient, future‑ready workforce. 🌍💼  \n\n#RemoteWork #FutureOfWork #HRStrategy #TalentAcquisition 
#Flexibility #GlobalTalent'}, next=(), config={'configurable': {'thread_id': 'Mukaram', 'checkpoint_ns': '', 
'checkpoint_id': '1f13ce7d-09e4-6bf1-8005-6312969dffe2'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, 
created_at='2026-04-20T18:36:16.034280+00:00', parent_config={'configurable': {'thread_id': 'Mukaram', 
'checkpoint_ns': '', 'checkpoint_id': '1f13ce7d-09cf-62f9-8004-7a1a683f370d'}}, tasks=(), interrupts=())